In [ ]:
# to we are handling PLAN and EXECUTE two brains, One job
"""Where ReAct reasons one step at a time and decides what to do as it goes,
Plan-and-execute separates thinking from doing entirely: one LLM call writes
the full plan upfront, and the second LLM executes each step in sequence.
The result is more predictable, easier to debug, and handles multi-step tasks
far better than ReAct alone.""" 

"""Today we work on the planner + executor architecture
The Planner: A detailed LLM call whose only job is to receive the user's question
and return a numbered checklist of steps. No tool calls, no answers, just a plan.
The output must be parseable : 1. Do x\n2. Do Y\n3. Do Z.
The Executor: A Second LLM call that receives one step at a time from the plan,
plus the results of all the previous steps as context, and executes it using my tool
roster from days 7-9
Step memory: As each step completes, it's result is appended to a completed_steps
list that the Executor receives as context for the next step. This is how the
executor builds on prior results without re-running anything.
Logging: Plugging my ReActTrace logger from Day9 into the executor so every step
is captured with the same structure"""


# importing necessary libraries

import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext

Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)
client = Groq()

@dataclass
class TraceStep:
    step_number: int
    timestamp: str
    raw_llm_output: str
    parsed_action: Optional[str]
    parsed_action_input: Optional[str]
    observation: Optional[str]
    is_final_answer: bool
    is_error_recovery: bool

@dataclass
class ReActTrace:
    question: str
    final_answer: Optional[str]
    total_steps: int
    success: bool
    started_at: str
    finished_at: str
    steps: list[TraceStep] = field(default_factory= list)

    def to_json(self, filepath: str):
        with open(filepath, "w") as f:
            json.dump(asdict(self), f, indent = 2)
        print(f"👍 Trace saved to {filepath}")

    @classmethod
    def from_json(cls, filepath: str):
        with open(filepath) as f:
            data = json.load(f)
        steps = [TraceStep(**s) for s in data.pop("steps")]
        trace = cls(**data)
        trace.steps = steps
        return trace


c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(find_dotenv())
# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')

🤖🛩️ Vector Store connection established ⚡


In [3]:
def search_local_docs(query:str) -> str:
    """Searches my local Knowledge base containing historical Apple 10-K financial documents
    (covering fiscal years up to 2024). Use this to retrieve historical sales, net revenue,
    and internal corporate performance figures.
    
    CRITICAL: Do not pass comparative or converstional questions here.
    Convert queries into strict financial line items, such as:
    - 'Apple consolidated statements of operations net sales' 
    - 'Apple total net slaes 2023 -2024' 
    - 'Summary of operations data'
    """

    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)
    return "\n\n".join([doc.node.get_content() for doc in results])

def get_doc_years(_: str = "") -> str:
    """Returns the list of available years in the 10k pdf"""
    return "Available years: 2021, 2022, 2023"

def calculator(expression: str) -> str:
    """Evaluates a basic math expression"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"
    
TOOLS = {
    "search_local_docs": search_local_docs,
    "get_doc_years": get_doc_years,
    "calculator": calculator,
}


In [ ]:
# THE PLANNER

PLANNER_SYSTEM_PROMPT = """You are a precise task planner.
Given a user question, produce a numbered step-by-step plan to answer it.

Rules:
- Each step must be a single, concrete action.
- Steps must use ONLY these tools: search_local_docs, get_doc_years, calculator
- If a step depends on the result of a previous step, say so explicitly
- Output ONLY the numbered list, no preamble, no explanation

Example output:
1. Use get_doc_years to get current years
2. Use calculator to count anything or perform any mathematical calculations from step 1
3. Use calculator to multiply the count from step 2 by current rates
"""

def plan(question: str) -> list[str]:
    response = client.chat.completions.create(
        model= "llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": f"Question: {question}"}
        ],
        temperature= 0,
    )

    raw_plan = response.choices[0].message.content
    print(f"\n 🦾 Plan generated:\n{raw_plan}\n")

    # parse numbered list into a Python list of strings
    steps = re.findall(r"\d+\.\s*(.+)", raw_plan)
    return steps

In [8]:
# THE EXECUTOR

EXECUTOR_SYSTEM_PROMPT = """You are a precise task executor.
You will be given one step to execute from a larger plan, plus the results of all
previous steps.

Rules:
- Execute ONLY the current step, nothing else
- Use exactly one tool call per step
- If the step requires a result from a previous step, extract it from the context provided
- Output your reasoning in one sentence, then your tool call.

Use this format:
Thought: <one sentence of reasoning>
Action: <tool name>
Action Input: <tool input>

"""

def execute_step(step: str, completed_steps: list[dict]) -> str:
    context = ""
    if completed_steps:
        context = "\n\nPrevious steps and their results:\n"
        for i, s in enumerate(completed_steps, 1):
            context += f"Step {i}: {s['step']}\nResult: {s['result']}\n\n"

    prompt = f"{context}Current step to execute: {step}"

    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [
            {"role": "system", "content": EXECUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature= 0,
        stop=["Observation"]
    )
    
    raw_output = response.choices[0].message.content
    print(f"Executor output:\n{raw_output}")

    # parse the action and run it
    action_match = re.search(r"Action:\s*(\w+)", raw_output)
    input_match = re.search(r"Action Input:\s*(.+)", raw_output)

    action = action_match.group(1).strip() if action_match else None
    action_input = input_match.group(1).strip() if input_match else ""

    if action and action in TOOLS:
        result = TOOLS[action](action_input)
        print(f"Observation: {result}\n")
        return result
    
    else:
        error = f"Error: tool '{action}' not found. Available: {list(TOOLS.keys())}"
        print(f"🤥 {error}\n")
        return error

In [9]:
# THE SYNTHESIZER

SYNTHESISER_PROMPT = """You are a precise answer writer.
Given a user question and the results of every step taken to answer it,
write a clear , concise final answer in 2-4 sentences.
Do not mention the steps or tools - just answer the question directly. 

"""

def synthesise(question: str, completed_steps: list[dict]) -> str:
    context = "\n".join(
        [f"Step {i + 1} result: {s['result']}" for i, s in enumerate(completed_steps)]
    )
    response = client.chat.completions.create(
        model= "llama-3.3-70b-versatile",
        messages = [
            {"role": "system", "content": SYNTHESISER_PROMPT},
            {"role": "user", "content": f"Question: {question}\n\nStep results:\n{context}"}
        ],
        temperature= 0,
    )
    return response.choices[0].message.content

In [10]:
# THE FULL PLAN AND EXECYTE RUNNER

def run_plan_and_execute(question: str) -> str:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    # Phase 1: Plan
    steps = plan(question= question)
    if not steps:
        return "Error: Planner produced no steps."
    
    # Phase 2: Execute each step
    completed_steps = []
    for i, step in enumerate(steps, 1):
        print(f"\n--- Executing Step {i}/{len(steps)}: {step} ---")
        result = execute_step(step, completed_steps= completed_steps)
        completed_steps.append({"step": step, "result": result})

    # Phase 3: Synthesise final answer
    final_answer = synthesise(question, completed_steps)
    print(f"\n{'='*60}")
    print(f"😁 Final Answer:\n{final_answer}")
    print(f"\n{'='*60}\n")

    # saving execution record to JSON
    record = {
        "question": question,
        "plan": steps,
        "completed_steps": completed_steps,
        "final_answer": final_answer,
        "timestamp": datetime.now().isoformat()
    }

    filename = f"traces/day10_plan_execute_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(record, f, indent= 2)
    print(f"📀 Saved to {filename}")

    return final_answer


# evaluation Questions, using those that failed or maxed out on day 8/9

# Qn 1. multi-step financial calculation
run_plan_and_execute(
    "Search the Apple 10-K for total net sales in fiscal year 2024, "
    "then calculate what percentage of that figure cam from iPhone revenue"
)

# qn 2. Requires Extracting and comparing two numbers
run_plan_and_execute(
    "Find Apple's total operating expenses for FY2024 and their total net income, "
    "then calculate the operating expense to net income ratio."
)

# Evaluation Question 3: Requires chaining three steps
run_plan_and_execute(
    "Search the 10-K for Apple's cash and cash equivalents at the end of FY2024, "
    "then find their total current liabilities, "
    "then calculate the current liquidity ratio between the two figures."

)


Question: Search the Apple 10-K for total net sales in fiscal year 2024, then calculate what percentage of that figure cam from iPhone revenue

 🦾 Plan generated:
1. Use search_local_docs to find the Apple 10-K document for fiscal year 2024
2. Use search_local_docs to find total net sales in the document from step 1
3. Use search_local_docs to find iPhone revenue in the document from step 1
4. Use calculator to divide iPhone revenue from step 3 by total net sales from step 2
5. Use calculator to multiply the result from step 4 by 100 to get the percentage


--- Executing Step 1/5: Use search_local_docs to find the Apple 10-K document for fiscal year 2024 ---
Executor output:
Thought: To find the Apple 10-K document for fiscal year 2024, we need to utilize the search_local_docs tool with the appropriate search query.
Action: search_local_docs
Action Input: Apple 10-K fiscal year 2024
Observation: at the dates and for the periods indicated.

Date: November 1, 2024

By: /s/ Timothy D. Co

"Apple's cash and cash equivalents at the end of FY2024 were $29,943 million. The total current liabilities were $176,392 million. The current liquidity ratio is 0.17, calculated by dividing cash and cash equivalents by total current liabilities."